<a href="https://colab.research.google.com/github/dianagironc1/proyecto-e10/blob/main/Copia_de_01_consultas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Consultas SQL · cuaderno del proyecto
**Estud-IA · Análisis de Datos Básico**

Corre las celdas en orden con el botón ▶ (o Shift + Enter). Si Colab muestra una advertencia porque el cuaderno viene de GitHub, elige **«Ejecutar de todos modos»**.

1. La celda 1 prepara todo. No hay que cambiar nada.
2. La celda 2 carga tu muestra.
3. La celda 3 cuenta las filas: tu primera **cifra de control**.
4. Desde la sesión 3: tus dos consultas, más abajo.

In [ ]:
# Celda 1 · Preparar (no cambies nada)
try:
    import duckdb
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb"])
    import duckdb
import re
import pandas as pd

def cargar(ruta):
    """Lee un CSV (coma o punto y coma) o un Excel. Acepta el enlace Raw o el de la página de GitHub."""
    ruta = re.sub(r"^https://github\.com/([^/]+)/([^/]+)/blob/", r"https://raw.githubusercontent.com/\1/\2/", ruta.strip())
    if ruta.lower().endswith((".xlsx", ".xls")):
        return pd.read_excel(ruta)
    for codificacion in ("utf-8-sig", "latin-1"):
        try:
            return pd.read_csv(ruta, sep=None, engine="python", encoding=codificacion)
        except UnicodeDecodeError:
            continue
        except pd.errors.ParserError:
            raise ValueError("No pude leerlo como tabla. Revisa que el enlace sea el de tu CSV (botón Raw).") from None
    raise ValueError("No pude leer el archivo: revisa que sea un CSV")

def sql(consulta):
    """Corre una consulta SQL sobre las tablas cargadas y muestra el resultado."""
    return duckdb.sql(consulta).df()

print("Listo. Sigue con la celda 2.")

Listo. Sigue con la celda 2.


## 2. Carga tu muestra

**Opción A (recomendada).** En GitHub abre tu CSV, copia la dirección de la página (o la del botón **Raw**) y pégala entre las comillas de `ENLACE`.

**Opción B.** Deja `ENLACE = ""` y corre la celda: aparece un botón para subir el archivo desde tu computador.

Con el enlace que viene (el caso de la papelería) deben salir **8 filas y 6 columnas**.

In [ ]:
# Celda 2 · Cargar la muestra
ENLACE = "https://github.com/dianagironc1/proyecto-e10/blob/main/datos/originales/sales_data_sample.csv"

if ENLACE:
    datos = cargar(ENLACE)
else:
    from google.colab import files
    subido = files.upload()
    datos = cargar(next(iter(subido)))

print(f"Cargué {len(datos)} filas y {len(datos.columns)} columnas.")
datos.head()

Cargué 2823 filas y 25 columnas.


,ORDERNUMBER,QUANTITYORDERED,PRICEEACH,ORDERLINENUMBER,SALES,ORDERDATE,STATUS,QTR_ID,MONTH_ID,YEAR_ID,...,ADDRESSLINE1,ADDRESSLINE2,CITY,STATE,POSTALCODE,COUNTRY,TERRITORY,CONTACTLASTNAME,CONTACTFIRSTNAME,DEALSIZE
0,10107,30.0,95.70,2.0,2871.00,2/24/2003 0:00,Shipped,1.0,2.0,2003.0,...,897 Long Airport Avenue,NaN,NYC,NY,10022,USA,NaN,Yu,Kwai,Small
1,10121,34.0,81.35,5.0,2765.90,5/7/2003 0:00,Shipped,2.0,5.0,2003.0,...,59 rue de l'Abbaye,NaN,Reims,NaN,51100,France,EMEA,Henriot,Paul,Small
2,10134,41.0,94.74,2.0,3884.34,7/1/2003 0:00,Shipped,3.0,7.0,2003.0,...,27 rue du Colonel Pierre Avia,NaN,Paris,NaN,75508,France,EMEA,Da Cunha,Daniel,Medium
3,10145,45.0,83.26,6.0,3746.70,8/25/2003 0:00,Shipped,3.0,8.0,2003.0,...,78934 Hillside Dr.,NaN,Pasadena,CA,90003,USA,NaN,Young,Julie,Medium
4,10159,49.0,100.00,14.0,5205.27,10/10/2003 0:00,Shipped,4.0,10.0,2003.0,...,7734 Strong St.,NaN,San Francisco,CA,NaN,USA,NaN,Brown,Julie,Medium


## 3. ¿Cuántas filas hay?

Compara con lo que escribiste en el diccionario, en «Filas en la muestra». Si no coincide, anótalo en la bitácora.

In [ ]:
# Celda 3 · Contar filas
sql("""
SELECT COUNT(*) AS filas
FROM datos
""")

,filas
0,2823


## 4. Los nombres exactos de tus columnas

Cópialos de aquí. Si un nombre tiene espacios, tildes o mayúsculas, escríbelo entre **comillas dobles**: `"Fecha del hecho"`. Los valores de texto van entre **comillas simples**: `'completada'`.

In [ ]:
# Celda 4 · Columnas de la muestra
list(datos.columns)

['ORDERNUMBER',
 'QUANTITYORDERED',
 'PRICEEACH',
 'ORDERLINENUMBER',
 'SALES',
 'ORDERDATE',
 'STATUS',
 'QTR_ID',
 'MONTH_ID',
 'YEAR_ID',
 'PRODUCTLINE',
 'MSRP',
 'PRODUCTCODE',
 'CUSTOMERNAME',
 'PHONE',
 'ADDRESSLINE1',
 'ADDRESSLINE2',
 'CITY',
 'STATE',
 'POSTALCODE',
 'COUNTRY',
 'TERRITORY',
 'CONTACTLASTNAME',
 'CONTACTFIRSTNAME',
 'DEALSIZE']

---
# Sesión 3 · Tus dos consultas

**Consulta 1 · Mirar y filtrar.** Primero cuenta todas las filas; después cuenta las que deja tu filtro; al final mira algunas. Cambia `estado`, `'completada'` y las columnas por las de tu muestra.

Con la papelería: 8 filas en total y 7 después del filtro.

In [ ]:
# Celda 5 · Consulta 1a: filas antes del filtro
sql("""
SELECT COUNT(*)
FROM datos
""")

,count_star()
0,2823


In [ ]:
# Celda 6 · Consulta 1b: filas después del filtro
sql("""
SELECT COUNT(*) AS filas_filtradas
FROM datos
WHERE status = 'Shipped'
""")

,filas_filtradas
0,1399


In [ ]:
# Celda 7 · Consulta 1c: mirar las filas filtradas, solo las columnas que necesitas
sql("""
SELECT ordernumber, quantityorder, priceeach, sales
FROM datos
WHERE status = 'shipped'
LIMIT 10
""")

,ORDERNUMBER,QUANTITYORDERED,PRICEEACH,SALES
0,10107,30.0,95.70,2871.00
1,10121,34.0,81.35,2765.90
2,10134,41.0,94.74,3884.34
3,10145,45.0,83.26,3746.70
4,10159,49.0,100.00,5205.27
5,10168,36.0,96.66,3479.76
6,10201,22.0,98.57,2168.54
7,10237,23.0,100.00,2333.12
8,10251,28.0,100.00,3188.64
9,10263,34.0,100.00,3676.76


**Consulta 2 · Resumir.** Cuenta cuántas filas hay en cada grupo. Cambia `estado` por la columna con la que quieres comparar (comuna, mes, categoría…).

**Cifra de control:** la suma de los grupos debe dar lo mismo que el total de la celda 5. Con la papelería: completada 7 + cancelada 1 = 8.

In [27]:
# Celda 8 · Consulta 2: resumen por grupo
sql("""
SELECT status, COUNT(*) AS filas
FROM datos
GROUP BY status
ORDER BY filas DESC
""")

,STATUS,filas
0,Shipped,1399
1,None,1315
2,On Hold,38
3,Cancelled,30
4,Resolved,29
5,In Process,9
6,Disputed,3


In [28]:
# Celda 9 · Cifra de control: ¿la suma de los grupos da el total?
sql("""
SELECT SUM(filas) AS suma_de_los_grupos
FROM (
    SELECT status, COUNT(*) AS filas
    FROM datos
    GROUP BY status
)
""")

,suma_de_los_grupos
0,2823.0


## Guarda tus consultas en GitHub

En tu repositorio, entra a `consultas/`, **Add file › Create new file**, ponle nombre (`01_mirar.sql`, `02_resumen.sql`), pega la consulta con tres comentarios arriba y haz commit con un mensaje que diga qué averiguaste:

```sql
-- Pregunta: ¿cuántas ventas se completaron?
-- Datos: datos/originales/ventas.csv (8 filas)
-- Cifra de control: 8 en total, 7 completadas
SELECT COUNT(*) AS filas_filtradas
FROM datos
WHERE estado = 'completada';
```

Mira el ejemplo resuelto en `consultas/` del repositorio de la papelería.

---
¿Algo falló? Regla de 3 y, si sigues bloqueado, un ticket en la **Cola de ayuda** con el mensaje de error copiado. Nunca pegues contraseñas ni datos personales.